# Part 5 – Final Integration, Documentation, and Submission

**Project:** Telco Customer Churn – Exploratory Data Analysis and Preprocessing  
**Final integrator:** Jamie Lopez

This notebook integrates Parts 1–4 into one reproducible workflow. It loads the original IBM Telco Customer Churn dataset, creates a 70/30 train/test split, performs EDA only on the training data, cleans and encodes the training data, scales continuous features, balances the training target, creates the `NumberOfServices` feature, exports the required files, and performs final verification.

**Important data rule:** the original dataset is never manually modified, and the testing data remains untouched by EDA, cleaning, scaling, oversampling, and feature engineering.


In [ ]:
# Install/import required libraries
!pip -q install imbalanced-learn

import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

print("Libraries imported successfully.")


## Part 1 – Dataset Setup and 70/30 Train/Test Split

The original dataset is downloaded only if it is not already present in the Colab session. The source is the IBM public copy of the Telco Customer Churn sample dataset, which corresponds to the Kaggle dataset documented below. The source file is read into pandas and is not edited in place.


In [ ]:
file_name = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
data_url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(data_url, file_name)
    print("Dataset downloaded and saved as", file_name)
else:
    print("Dataset file already present:", file_name)

df = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Original dataset shape:", df.shape)
display(df.head())


In [ ]:
# Inspect the original dataset without modifying it
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nDuplicate rows:", df.duplicated().sum())
print("\nMissing/null values:")
display(df.isnull().sum())


In [ ]:
# Separate features and target, then create the required 70/30 split.
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

train_data = X_train.copy()
train_data["Churn"] = y_train

test_data = X_test.copy()
test_data["Churn"] = y_test

print("Training shape:", train_data.shape)
print("Testing shape:", test_data.shape)
print("Training percentage:", round(len(train_data) / len(df) * 100, 2), "%")
print("Testing percentage:", round(len(test_data) / len(df) * 100, 2), "%")

assert len(train_data) + len(test_data) == len(df)
assert len(test_data) / len(df) == 0.30
print("\nSUCCESS: 70/30 split verified.")


In [ ]:
# Save the untouched train/test split files
train_data.to_csv("telco_training_data.csv", index=False)
test_data.to_csv("telco_testing_data.csv", index=False)

print("Training and testing datasets saved successfully.")


## Part 2 – Exploratory Data Analysis (Training Data Only)

EDA is performed on a copy of the 70% training data. The testing data is not used for EDA. This preserves the testing set as an untouched holdout for later modeling/evaluation.


In [ ]:
eda_df = train_data.copy()

print("EDA is being performed on TRAINING data only.")
print("Training data shape:", eda_df.shape)

display(eda_df.head(10))

print("Training dimensions:", eda_df.shape)
print("\nNumerical summary:")
display(eda_df.describe())

print("\nCategorical summary:")
display(eda_df.describe(include="object"))

print("\nData types:")
display(eda_df.dtypes)

print("\nMissing/null values:")
display(eda_df.isnull().sum())


In [ ]:
# Investigate hidden blanks and unusual values without changing the training data.
blank_counts = eda_df.apply(lambda col: (col.astype(str).str.strip() == "").sum())
print("Blank/whitespace entries:")
display(blank_counts[blank_counts > 0])

total_charges_numeric = pd.to_numeric(eda_df["TotalCharges"], errors="coerce")
print("TotalCharges values that cannot be converted to numbers:",
      total_charges_numeric.isna().sum())

print("\nRows with blank/invalid TotalCharges:")
display(eda_df.loc[
    total_charges_numeric.isna(),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
])

print("Tenure = 0:", (eda_df["tenure"] == 0).sum())
print("Negative tenure:", (eda_df["tenure"] < 0).sum())
print("Negative MonthlyCharges:", (eda_df["MonthlyCharges"] < 0).sum())
print("Duplicate rows:", eda_df.duplicated().sum())
print("Duplicate customer IDs:", eda_df["customerID"].duplicated().sum())


In [ ]:
# Churn distribution
churn_counts = eda_df["Churn"].value_counts()
churn_percent = eda_df["Churn"].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    "Count": churn_counts,
    "Percentage (%)": churn_percent.round(2)
})
display(churn_summary)

plt.figure(figsize=(6,4))
bars = plt.bar(churn_counts.index, churn_counts.values)
for bar, pct in zip(bars, churn_percent.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
             f"{int(bar.get_height())} ({pct:.1f}%)", ha="center")
plt.title("Churn: Yes vs. No (Training Data)")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# Key numerical variables
key_numeric = eda_df[["tenure", "MonthlyCharges"]].copy()
key_numeric["TotalCharges"] = total_charges_numeric
key_numeric["Churn"] = eda_df["Churn"]

print("Average values by Churn:")
display(key_numeric.groupby("Churn").mean().round(2))

print("Median values by Churn:")
display(key_numeric.groupby("Churn").median().round(2))

print("Correlation:")
display(key_numeric[["tenure", "MonthlyCharges", "TotalCharges"]].corr().round(2))

# Before/after-style EDA views
fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].hist(eda_df["tenure"], bins=24, edgecolor="black")
axes[0].set_title("Tenure Distribution – All Training Customers")
axes[0].set_xlabel("Tenure (months)")
axes[0].set_ylabel("Customers")

axes[1].hist(eda_df.loc[eda_df["Churn"]=="No","tenure"], bins=24, alpha=.6, label="No", edgecolor="black")
axes[1].hist(eda_df.loc[eda_df["Churn"]=="Yes","tenure"], bins=24, alpha=.6, label="Yes", edgecolor="black")
axes[1].set_title("Tenure Distribution by Churn")
axes[1].set_xlabel("Tenure (months)")
axes[1].set_ylabel("Customers")
axes[1].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].hist(eda_df["MonthlyCharges"], bins=30, edgecolor="black")
axes[0].set_title("Monthly Charges – All Training Customers")
axes[0].set_xlabel("Monthly Charges ($)")
axes[0].set_ylabel("Customers")

axes[1].boxplot([
    eda_df.loc[eda_df["Churn"]=="No","MonthlyCharges"],
    eda_df.loc[eda_df["Churn"]=="Yes","MonthlyCharges"]
])
axes[1].set_xticks([1,2])
axes[1].set_xticklabels(["No","Yes"])
axes[1].set_title("Monthly Charges by Churn")
axes[1].set_xlabel("Churn")
axes[1].set_ylabel("Monthly Charges ($)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.hist(total_charges_numeric.dropna(), bins=30, edgecolor="black")
plt.title("Total Charges Distribution – Training Data")
plt.xlabel("Total Charges ($)")
plt.ylabel("Customers")
plt.show()


In [ ]:
# Categorical churn-rate analysis
def plot_churn_by_category(column, title, rotate=0):
    table = pd.crosstab(eda_df[column], eda_df["Churn"])
    ax = table.plot(kind="bar", figsize=(8,4), edgecolor="black")
    rates = table["Yes"] / table.sum(axis=1) * 100
    for i, rate in enumerate(rates):
        ax.text(i, table.iloc[i].max() + 30, f"{rate:.1f}%", ha="center")
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.set_ylabel("Number of Customers")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()

for col in ["Contract", "InternetService", "PaymentMethod"]:
    counts = eda_df[col].value_counts()
    rates = (pd.crosstab(eda_df[col], eda_df["Churn"], normalize="index")["Yes"] * 100).round(1)
    print(f"\n{col}:")
    display(pd.DataFrame({"Customers": counts, "Churn Rate (%)": rates}).sort_values("Churn Rate (%)", ascending=False))

plot_churn_by_category("Contract", "Contract Type vs. Churn – Training Data")
plot_churn_by_category("InternetService", "Internet Service vs. Churn – Training Data")
plot_churn_by_category("PaymentMethod", "Payment Method vs. Churn – Training Data", rotate=15)

tenure_groups = pd.cut(
    eda_df["tenure"],
    bins=[-1,12,24,48,72],
    labels=["0 to 12 months","13 to 24 months","25 to 48 months","49 to 72 months"]
)
tenure_churn_rate = pd.crosstab(
    tenure_groups, eda_df["Churn"], normalize="index"
)["Yes"] * 100

plt.figure(figsize=(8,4))
bars = plt.bar(tenure_churn_rate.index.astype(str), tenure_churn_rate.values)
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f"{bar.get_height():.1f}%", ha="center")
plt.title("Churn Rate by Tenure Group – Training Data")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.ylim(0, 60)
plt.show()


### EDA Findings

The training data shows class imbalance, hidden blank `TotalCharges` values, and strong churn patterns associated with short tenure, month-to-month contracts, fiber optic service, electronic-check payment, and higher monthly charges. `customerID` is an identifier rather than a predictive feature. These observations guide the cleaning and preprocessing steps that follow.


In [ ]:
# Verify EDA did not modify the original training data.
assert train_data.equals(X_train.assign(Churn=y_train))
assert test_data.equals(X_test.assign(Churn=y_test))
print("SUCCESS: Training and testing split objects remain unchanged by EDA.")


## Part 3 – Data Cleaning and Categorical Encoding

Cleaning begins from a copy of the training data. The testing data is intentionally excluded from this process.


In [ ]:
clean_df = train_data.copy()

# Convert TotalCharges to numeric. Blank values become NaN in this copy.
clean_df["TotalCharges"] = pd.to_numeric(clean_df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after conversion:", clean_df["TotalCharges"].isnull().sum())

# The affected customers have 0 months of tenure, so their accumulated charges are 0.
clean_df["TotalCharges"] = clean_df["TotalCharges"].fillna(0)

print("Missing TotalCharges after filling:", clean_df["TotalCharges"].isnull().sum())

print("Duplicate rows before removing customerID:", clean_df.duplicated().sum())
print("Duplicate customer IDs:", clean_df["customerID"].duplicated().sum())


In [ ]:
# Remove identifier
clean_df = clean_df.drop(columns=["customerID"])

binary_columns = [
    "gender", "Partner", "Dependents",
    "PhoneService", "PaperlessBilling"
]

one_hot_columns = [
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaymentMethod"
]

print("Data BEFORE encoding:")
display(clean_df.head())

binary_mapping = {
    "Yes": 1, "No": 0,
    "Male": 1, "Female": 0
}

for column in binary_columns:
    clean_df[column] = clean_df[column].map(binary_mapping)

clean_df = pd.get_dummies(
    clean_df,
    columns=one_hot_columns,
    drop_first=False,
    dtype=int
)

clean_df["Churn"] = clean_df["Churn"].map({"Yes": 1, "No": 0})

print("Data AFTER encoding:")
display(clean_df.head())

remaining_objects = clean_df.select_dtypes(include="object").columns.tolist()
print("Remaining object columns:", remaining_objects)
print("Missing values:", clean_df.isnull().sum().sum())

assert len(remaining_objects) == 0
assert clean_df.isnull().sum().sum() == 0
print("SUCCESS: Cleaning and encoding completed.")


In [ ]:
# Duplicate review after removing customerID
print("Exact duplicate feature rows after removing customerID:", clean_df.duplicated().sum())
print("These rows are retained because different customers can legitimately have identical feature values.")

clean_df.to_csv("telco_churn_cleaned_training.csv", index=False)
print("Saved: telco_churn_cleaned_training.csv")


## Part 4 – Scaling, Normalization, Balancing, and Feature Engineering

Only the cleaned **training** dataset is used here. The testing dataset is still untouched.

Continuous variables (`tenure`, `MonthlyCharges`, and `TotalCharges`) are standardized. Binary/one-hot variables remain 0/1. The Churn target is not scaled. Random oversampling is then applied only to the training data.


In [ ]:
scale_columns = ["tenure", "MonthlyCharges", "TotalCharges"]

before_scaling = clean_df[scale_columns].copy()
ml_df = clean_df.copy()

scaler = StandardScaler()
ml_df[scale_columns] = scaler.fit_transform(clean_df[scale_columns])

print("Before scaling:")
display(before_scaling.head(10))

print("After scaling:")
display(ml_df[scale_columns].head(10))

fig = plt.figure(figsize=(9,5))
before_scaling.plot(kind="box", figsize=(9,5))
plt.title("Continuous Features Before Scaling")
plt.ylabel("Original Values")
plt.xlabel("Features")
plt.show()

ml_df[scale_columns].plot(kind="box", figsize=(9,5))
plt.title("Continuous Features After Standard Scaling")
plt.ylabel("Standardized Values")
plt.xlabel("Features")
plt.show()

print("Scaled feature means (approximately 0):")
display(ml_df[scale_columns].mean().round(4))


In [ ]:
# Examine and balance Churn in the training data only
churn_counts = ml_df["Churn"].value_counts().sort_index()
churn_percent = ml_df["Churn"].value_counts(normalize=True).sort_index() * 100

display(pd.DataFrame({
    "Count": churn_counts,
    "Percentage": churn_percent.round(2)
}))

churn_counts.plot(kind="bar", figsize=(7,5))
plt.title("Churn Class Distribution Before Oversampling")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.show()

X_bal_input = ml_df.drop(columns=["Churn"])
y_bal_input = ml_df["Churn"]

ros = RandomOverSampler(random_state=42)
X_balanced, y_balanced = ros.fit_resample(X_bal_input, y_bal_input)

print("Class counts before balancing:")
print(y_bal_input.value_counts().sort_index())

print("\nClass counts after balancing:")
print(y_balanced.value_counts().sort_index())

assert y_balanced.value_counts().nunique() == 1


In [ ]:
# Recombine balanced training data
processed_df = X_balanced.copy()
processed_df["Churn"] = y_balanced.values

# Feature engineering: count optional services
service_columns = [
    "OnlineSecurity_Yes",
    "OnlineBackup_Yes",
    "DeviceProtection_Yes",
    "TechSupport_Yes",
    "StreamingTV_Yes",
    "StreamingMovies_Yes"
]

processed_df["NumberOfServices"] = processed_df[service_columns].sum(axis=1)

print("NumberOfServices distribution:")
display(processed_df["NumberOfServices"].value_counts().sort_index())

print("Final processed training dataset shape:", processed_df.shape)
display(processed_df.head())


In [ ]:
# Final verification and exports
print("Final processed training shape:", processed_df.shape)
print("Missing values:", processed_df.isnull().sum().sum())

non_numeric = processed_df.select_dtypes(exclude="number").columns.tolist()
print("Non-numeric columns:", non_numeric)

print("Final Churn distribution:")
print(processed_df["Churn"].value_counts().sort_index())

assert processed_df.isnull().sum().sum() == 0
assert len(non_numeric) == 0
assert processed_df["Churn"].value_counts().nunique() == 1

processed_df.to_csv("telco_churn_final_processed_training.csv", index=False)

print("\nSUCCESS: Final processed training dataset exported.")


## Final Integrity Checks

These checks explicitly verify the most important Part 5 requirements:

1. The split is 70/30.
2. EDA and preprocessing used the training data.
3. The test set was not modified by the workflow.
4. The original source dataframe remains separate from the processed training dataframe.
5. The final processed training data has no missing or non-numeric values.
6. The balanced training target has equal class counts.


In [ ]:
# Re-read the saved testing data and compare it to the original test split.
saved_test = pd.read_csv("telco_testing_data.csv")

assert saved_test.shape == test_data.shape
assert saved_test.equals(test_data.reset_index(drop=True))

print("TEST SET CHECK: PASS")
print("- Testing data was created before cleaning/scaling/oversampling.")
print("- Testing data was not used for EDA.")
print("- Testing data was not used to fit the scaler.")
print("- Testing data was not oversampled.")
print("- Saved testing file matches the untouched test split.")

print("\nTRAINING CHECK: PASS")
print("- 70% training split verified.")
print("- Cleaning/encoding performed on training data.")
print("- Scaling performed on training data.")
print("- Oversampling performed on training data only.")
print("- NumberOfServices created on balanced training data.")

print("\nFILES CREATED:")
for filename in [
    "telco_training_data.csv",
    "telco_testing_data.csv",
    "telco_churn_cleaned_training.csv",
    "telco_churn_final_processed_training.csv"
]:
    print(" -", filename, "->", os.path.exists(filename))


## Kaggle Dataset Source

**Kaggle:** IBM Telco Customer Churn sample dataset  
https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Public IBM copy used by the notebook:**  
https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv

The notebook downloads the public IBM copy only when the CSV is not already present. The original source file is read but never manually modified.


## Contribution Journal – Person 5: Jamie Lopez

For Part 5, I integrated Parts 1–4 into one final Colab notebook and organized the sections in the required order. I reviewed the workflow from the initial dataset download through the final processed training dataset and added final verification checks so the notebook can run from the first cell to the last cell without requiring the original dataset to be manually edited.

I verified the 70/30 train/test split, confirmed that EDA and preprocessing were performed using the training data rather than the untouched testing data, and checked that the testing file remained unchanged. I also made sure the notebook contains Matplotlib graphs, before/after scaling examples, code comments, explanations, the Kaggle dataset source, and exports for the training, testing, cleaned, and final processed datasets.

I prepared the final project files for submission and organized the documentation needed for the group's repository. I also checked the final processed training data for missing values, non-numeric columns, and balanced Churn classes. This completed the final integration and documentation responsibilities for Part 5.
